# Integration of `duckdb` in Python

This is again just the prologue to allow the notebooks running standalone in a local Jupyter installation and in Google Colab.

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    if not os.path.isdir("/content/oreilly-duckdb"):
        os.system("git clone https://github.com/datanizing/oreilly-duckdb")
    os.system("pip install -U duckdb")
    data = "/content/oreilly-duckdb/data"
except:
    IN_COLAB = False
    data = "data"

In [ ]:
import duckdb

## `DataFrame` integration

As you have seen in the previous notebooks, `duckdb` results can
be converted to a `DataFrame` very easily, irrespective of whether
you are using `pandas` or `polars`.

Interestingly, it also works the other way round. If you have
a `DataFrame` in Python, you can query it like a `duckdb` table.

Let's create a `DataFrame` in `pandas` via reading the weather
data from CSV:

In [ ]:
import pandas as pd
df = pd.read_csv(f'{data}/weatherAUS.csv.zst')
df

As you can see, `pandas` does not show the data types in the
`DataFrame`, but we can get them manually:

In [ ]:
df.dtypes

Now we try to use `duckdb` to write an SQL query for
the `DataFrame`:

In [ ]:
duckdb.sql(f"SELECT * FROM df").pl()

This looks like it *can't' work - but it does. What's happening?

`duckdb` and Python share the same process and the same data. This
means that `duckdb` can also access Python objects - like a `DataFrame`.

That's very nice as you don't have to copy data or persist it for
later querying. Moreover, you don't have to remember the (very
specific) syntax for selection, aggregation etc. in the various
frameworks but you can use `duckdb` and SQL to facilitate that.

As `duckdb` is framework-agnostic, this should also work with a `polars`
`DataFrame`. First load the CSV file into the `DataFrame`:

In [ ]:
import polars as pl
dfp = pl.read_csv(f'{data}/weatherAUS.csv.zst')
dfp

`polars` show the data types in the `DataFrame`. Querying
it via `duckdb` and converting it back to the `DataFrame`
show virtually no change:

In [ ]:
duckdb.sql(f"SELECT * FROM dfp").pl()

Check if the results are really identical:

In [ ]:
dfp.equals(duckdb.sql(f"SELECT * FROM dfp").pl())

**Bottom line**: You can work with a `DataFrame` like a table in `duckdb`.

## Using `duckdb` to read data

`pandas` and `polars` offer a lot of possibilities how to read data.
However, not all of them can read compressed data, XML data
might be complicated and using `glob` expressions is not supported.

Therefore, it is often much more convenient to use `duckdb` for
reading that kind of data. You can either use an SQL statement
for that or make use of the Python API directly. Try this for
reading the compressed CSV file:

In [ ]:
duckdb.read_csv(f'{data}/weatherAUS.csv.zst').pl()

This is working well for smaller datasets, but if everything is
directly converted to a `DataFrame`, it will eat up a lot of
memory.

Therefore, it is often better to *keep* that data inside `duckdb`.
Let's assign the result of the `read_csv` to a Python variable
and take a look at the content of this object:

In [ ]:
dfd = duckdb.read_csv(f'{data}/weatherAUS.csv.zst')

In [ ]:
type(dfd)

This is not the actual data like we might be tempted
to believe! If we output the variable, the data will
be shown though in box form. Converting it to a `DataFrame`
transfers the data to Python using its internal format
and using potentially a lot of RAM.

Until now, we have made use of the fact that a relation
object will always show its associated data.

But we can also use this to e.g. calculate aggregations
which obviously do not take a lot of memory:

In [ ]:
dfd.count("*")

`duckdb`'s Python interface offers many different functions for 
working with the `DuckDBPyRelation`. The most common are:

* `filter`: Include only rows that satisfy a condition (like the conditions after `WHERE` in SQL).
* `project`: Include only specific columns (like those after `SELECT` in SQL).
* `limit`: Same functionality as `LIMIT` in SQL or `head` in `pandas`/`polars`.
* `aggregate`: Works like `GROUP BY` in SQL.
* `order`: Same functionality as `ORDER`in SQL.

In this course, we will not focus on this API. Apart from
the fact that this is Python, it does not offer tremendous
advantages compared to SQL. On the contrary, you have to
remember these idioms in another language.

Actually more useful is the functionality of storing 
that relation in a table. This can also be achieved with
the Python API:

In [ ]:
dfd.to_table("weather")

This table can now be queried like any other `duckdb`
table:

In [ ]:
duckdb.sql("SELECT * FROM weather WHERE MinTemp>30").pl()

If you want, you could also `ATTACH` a file
and save it in a persistent table.

## Extending `duckdb` with Python functions

If you have worked with relational databases before,
you probably have encountered their specific languages
for writing functions and adding functionality to them.
This can be immensely useful, save time and increase
performance.

In `duckdb`, you can write your function in Python!

If you take a look at the table above, you can see that
some locations are missing spaces (like *MelbourneAirport*).
Maybe that's due to some restriction in the data format.

If you want to create a professional diagram, adding these
spaces would be an excellent idea. Althout it *is* possible
in `duckdb`, we will try to write a Python function for
that. This has several advantages like a language you already
know and much easier testing.

We will start by working with the `re` library for 
regular expressions. We will match a lowercase
character and a uppercase character after that:

In [ ]:
import re
re.match("[a-z][A-Z]", "MelbourneAirport")

It does not match 🤔.

`re` is a bit special, `match` only works at the beginning
of a string. To also match in the middle, we have to use
`search`:

In [ ]:
re.search("[a-z][A-Z]", "MelbourneAirport")

Excellent, now we want to insert a space in between.
Turns out that this is not so easy, we have to use
groups for matching so we can insert the matched
letters again:

In [ ]:
re.sub(r"([a-z])([A-Z])", r"\1 \2", "MelbourneAirport")

Great, let's write a function which accomplished this:

In [ ]:
def insert_spaces(field:str) -> str:
  if field:
    return re.sub(r"([a-z])([A-Z])", r"\1 \2", field)
  else:
    return field

And test it:

In [ ]:
insert_spaces("MelbourneAirport")

Now we still have to integrate it in `duckdb`:

In [ ]:
duckdb.create_function('insert_spaces', insert_spaces)

Looks like it has worked, check the result:

In [ ]:
duckdb.sql("SELECT DISTINCT(insert_spaces(Location)) FROM weather").pl()

Excellent, we now have a function which "normalizes"
the names and makes them available in a way which would
also be suitable for a presentation.